In [ ]:
"""
GeoSDI Geothermal v2.0 — Data Exploration Notebook
===================================================
Eksplorasi pertama: node geothermal + batas administrasi Indonesia.
"""

import sys
from pathlib import Path

# Tambahkan root project ke path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

from src.shared.config import get_settings
from src.shared.logger import setup_logging, get_logger

# Setup logging
setup_logging()
log = get_logger(__name__)
settings = get_settings()

log.info("Notebook started", extra={"data_path": str(settings.data_raw_path)})

# Konfigurasi display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (14, 8)

print(f"📁 Data raw path : {settings.data_raw_path}")
print(f"📁 Logs path     : {settings.logs_path}")
print(f"🌏 Environment   : {settings.app_env}")

In [ ]:
# Baca CSV node geothermal
nodes_path = settings.data_raw_path / "geothermal_nodes.csv"
nodes = pd.read_csv(nodes_path)

print(f"📊 Jumlah node: {len(nodes)}")
print(f"📋 Kolom: {list(nodes.columns)}")
print("\n🔍 Preview 5 baris pertama:")
nodes.head()

In [ ]:
print("=" * 60)
print("STRUKTUR DATA")
print("=" * 60)
nodes.info()

print("\n" + "=" * 60)
print("STATISTIK DESKRIPTIF")
print("=" * 60)
nodes.describe(include="all")

In [ ]:
missing = nodes.isna().sum()
missing_pct = (missing / len(nodes) * 100).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct,
}).sort_values("missing_pct", ascending=False)

print("🔍 Laporan Missing Values:")
missing_report[missing_report["missing_count"] > 0]

In [ ]:
# Baca GeoJSON provinsi (admin1 = level provinsi)
admin1_path = settings.data_raw_path / "idn_admin1.geojson"

log.info("Loading admin1 geojson", extra={"path": str(admin1_path)})
indonesia = gpd.read_file(admin1_path)

print(f"🗺️  Jumlah fitur: {len(indonesia)}")
print(f"📋 Kolom: {list(indonesia.columns)}")
print(f"📍 CRS: {indonesia.crs}")
print("\n🔍 Preview:")
indonesia.head()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

indonesia.plot(
    ax=ax,
    color="lightblue",
    edgecolor="navy",
    linewidth=0.5,
    alpha=0.7,
)

ax.set_title(
    "🇮🇩 Batas Administrasi Indonesia (Level Provinsi)",
    fontsize=16,
    fontweight="bold",
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

log.info("Peta Indonesia berhasil digambar", extra={"n_provinces": len(indonesia)})

In [ ]:
# Cek dulu kolom apa saja yang ada di nodes
print("Kolom di nodes:", list(nodes.columns))

# Deteksi otomatis kolom latitude & longitude
lat_candidates = [c for c in nodes.columns if c.lower() in ("lat", "latitude", "y")]
lon_candidates = [c for c in nodes.columns if c.lower() in ("lon", "lng", "longitude", "x")]

print(f"Kolom latitude  terdeteksi: {lat_candidates}")
print(f"Kolom longitude terdeteksi: {lon_candidates}")

# Jika nama kolom berbeda, sesuaikan di sini:
LAT_COL = lat_candidates[0] if lat_candidates else None
LON_COL = lon_candidates[0] if lon_candidates else None

if LAT_COL and LON_COL:
    # Konversi ke GeoDataFrame
    geometry = [Point(xy) for xy in zip(nodes[LON_COL], nodes[LAT_COL])]
    nodes_gdf = gpd.GeoDataFrame(nodes, geometry=geometry, crs="EPSG:4326")

    print(f"\n✅ Berhasil konversi {len(nodes_gdf)} node ke GeoDataFrame")
    print(f"📍 CRS: {nodes_gdf.crs}")
    print("\n🔍 Preview geometry:")
    print(nodes_gdf[[LAT_COL, LON_COL, "geometry"]].head())
else:
    print("\n⚠️  Kolom koordinat tidak terdeteksi otomatis.")
    print("    Silakan cek nama kolom di output di atas dan sesuaikan.")

In [ ]:
if LAT_COL and LON_COL:
    fig, ax = plt.subplots(figsize=(18, 11))

    # Layer 1: Batas provinsi
    indonesia.plot(
        ax=ax,
        color="#f0f4f8",
        edgecolor="#4a5568",
        linewidth=0.4,
        alpha=0.6,
    )

    # Layer 2: Node geothermal
    nodes_gdf.plot(
        ax=ax,
        color="red",
        markersize=80,
        edgecolor="darkred",
        linewidth=1.2,
        alpha=0.85,
        label=f"Geothermal Nodes (n={len(nodes_gdf)})",
        zorder=5,
    )

    # Label tiap node
    for idx, row in nodes_gdf.iterrows():
        if "name" in row and pd.notna(row["name"]):
            ax.annotate(
                row["name"],
                xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
                color="darkred",
                fontweight="bold",
            )

    ax.set_title(
        "🌋 GeoSDI Geothermal — Sebaran Node Panas Bumi Indonesia",
        fontsize=18,
        fontweight="bold",
        pad=20,
    )
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.legend(loc="lower right", fontsize=11, frameon=True)
    ax.grid(True, alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig(
        settings.data_processed_path / "peta_node_geothermal.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

    log.info(
        "Peta node geothermal tersimpan",
        extra={"n_nodes": len(nodes_gdf), "lat_col": LAT_COL, "lon_col": LON_COL},
    )
    print(f"\n💾 Peta tersimpan di: {settings.data_processed_path / 'peta_node_geothermal.png'}")

In [ ]:
# Cek apakah ada kolom provinsi di data nodes
province_candidates = [
    c for c in nodes.columns
    if any(k in c.lower() for k in ("prov", "province", "wilayah", "region"))
]

if province_candidates:
    prov_col = province_candidates[0]
    print(f"📍 Kolom provinsi: {prov_col}")

    prov_stats = nodes[prov_col].value_counts().reset_index()
    prov_stats.columns = ["province", "count"]
    prov_stats = prov_stats.sort_values("count", ascending=False)

    print(f"\n📊 Distribusi Node per Provinsi:")
    print(prov_stats.to_string(index=False))

    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(
        prov_stats["province"],
        prov_stats["count"],
        color="steelblue",
        edgecolor="navy",
    )
    ax.set_xlabel("Jumlah Node", fontsize=12)
    ax.set_title("Distribusi Node Geothermal per Provinsi", fontsize=14, fontweight="bold")
    ax.invert_yaxis()
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Tidak ada kolom provinsi di data. Kita akan join dengan GeoJSON nanti.")

In [ ]:
# Simpan data yang sudah bersih
if LAT_COL and LON_COL:
    output_path = settings.data_processed_path / "geothermal_nodes.geojson"
    nodes_gdf.to_file(output_path, driver="GeoJSON")

    log.info("Data bersih tersimpan", extra={"path": str(output_path)})
    print(f"\n✅ Data bersih tersimpan di:")
    print(f"   {output_path}")
    print(f"\n📊 Total node: {len(nodes_gdf)}")
    print(f"📊 Kolom: {list(nodes_gdf.columns)}")